# 06 — Presentation-Ready Goal Regression Model Comparison

This notebook compares multiple regression models for football goal prediction and creates presentation-ready evaluation plots.

## Project Setup

The modelling dataset and output folders are prepared.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from pandas.api.types import is_numeric_dtype
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, PoissonRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except Exception:
    LIGHTGBM_AVAILABLE = False

PROJECT_DIR = Path("..").resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "output"
COMPARISON_DIR = OUTPUT_DIR / "regression_model_comparison_presentation"
TABLES_DIR = COMPARISON_DIR / "tables"
FIGURES_DIR = COMPARISON_DIR / "figures"
MODELS_DIR = COMPARISON_DIR / "models"

for directory in [TABLES_DIR, FIGURES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Processed data folder:", PROCESSED_DIR)
print("Comparison output folder:", COMPARISON_DIR)
print("XGBoost available:", XGBOOST_AVAILABLE)
print("LightGBM available:", LIGHTGBM_AVAILABLE)


## Load Dataset

The feature-engineered training dataset is loaded from the main project.

In [ ]:
model_dataset = pd.read_csv(PROCESSED_DIR / "model_dataset.csv")
model_dataset["date"] = pd.to_datetime(model_dataset["date"], errors="coerce")

print("Model dataset shape:", model_dataset.shape)
print("Date range:", model_dataset["date"].min(), "to", model_dataset["date"].max())

display(model_dataset.head())
display(model_dataset[["team_a_goals", "team_b_goals"]].describe())


## Goal Data Overview

Goal distributions are visualized before model comparison.

In [ ]:
goal_overview = model_dataset.copy()
goal_overview["total_goals"] = goal_overview["team_a_goals"] + goal_overview["team_b_goals"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(goal_overview["total_goals"], bins=20)
ax.set_title("Distribution of Total Goals per Match")
ax.set_xlabel("Total goals")
ax.set_ylabel("Number of matches")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_total_goals_distribution.png", dpi=150)
plt.show()

avg_goals_per_year = goal_overview.groupby(goal_overview["date"].dt.year)["total_goals"].mean()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(avg_goals_per_year.index, avg_goals_per_year.values, marker="o")
ax.set_title("Average Total Goals per Match by Year")
ax.set_xlabel("Year")
ax.set_ylabel("Average total goals")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_average_goals_per_year.png", dpi=150)
plt.show()


## Feature and Target Selection

The same feature set is used to predict Team A goals and Team B goals.

In [ ]:
target_goals_a = "team_a_goals"
target_goals_b = "team_b_goals"

identifier_columns = ["team_a", "team_b", "date"]
target_columns = ["result", "team_a_goals", "team_b_goals"]

feature_columns = [
    col for col in model_dataset.columns
    if col not in identifier_columns + target_columns
]

X = model_dataset[feature_columns].copy()
y_goals_a = model_dataset[target_goals_a].copy()
y_goals_b = model_dataset[target_goals_b].copy()

numeric_features = [col for col in X.columns if is_numeric_dtype(X[col])]
categorical_features = [col for col in X.columns if col not in numeric_features]

print("Number of features:", len(feature_columns))
print("Number of numeric features:", len(numeric_features))
print("Categorical features:", categorical_features)


## Chronological Train-Test Split

A time-based split avoids evaluating the model on older matches after training on newer ones.

In [ ]:
model_dataset_sorted = model_dataset.sort_values("date").reset_index(drop=True)
split_index = int(len(model_dataset_sorted) * 0.8)

train_data = model_dataset_sorted.iloc[:split_index].copy()
test_data = model_dataset_sorted.iloc[split_index:].copy()

X_train = train_data[feature_columns].copy()
X_test = test_data[feature_columns].copy()

y_train_a = train_data[target_goals_a].copy()
y_test_a = test_data[target_goals_a].copy()

y_train_b = train_data[target_goals_b].copy()
y_test_b = test_data[target_goals_b].copy()

print("Train rows:", len(train_data))
print("Test rows:", len(test_data))
print("Train date range:", train_data["date"].min(), "to", train_data["date"].max())
print("Test date range:", test_data["date"].min(), "to", test_data["date"].max())


## Preprocessing Pipeline

Numeric features are imputed and scaled. Categorical features are one-hot encoded.

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

print("Preprocessing pipeline created.")


## Regression Models

Several regression models are compared for score prediction.

In [ ]:
regression_models = {
    "Dummy Mean Baseline": DummyRegressor(strategy="mean"),
    "Ridge Regression": Ridge(alpha=1.0),
    "Poisson Regression": PoissonRegressor(alpha=0.1, max_iter=1000),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),
    "Gradient Boosting Regressor": GradientBoostingRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),
}

if XGBOOST_AVAILABLE:
    regression_models["XGBoost Regressor"] = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )
else:
    print("XGBoost is not installed. XGBoost Regressor will be skipped.")

if LIGHTGBM_AVAILABLE:
    regression_models["LightGBM Regressor"] = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
else:
    print("LightGBM is not installed. LightGBM Regressor will be skipped.")

print("Models included:")
for model_name in regression_models:
    print(" -", model_name)


## Train and Evaluate Models

Each model predicts goals for Team A and Team B separately.

In [ ]:
def evaluate_regression(y_true_a, pred_a, y_true_b, pred_b, model_name):
    rounded_a = np.round(pred_a).astype(int)
    rounded_b = np.round(pred_b).astype(int)

    mae_a = mean_absolute_error(y_true_a, pred_a)
    mae_b = mean_absolute_error(y_true_b, pred_b)

    rmse_a = np.sqrt(mean_squared_error(y_true_a, pred_a))
    rmse_b = np.sqrt(mean_squared_error(y_true_b, pred_b))

    r2_a = r2_score(y_true_a, pred_a)
    r2_b = r2_score(y_true_b, pred_b)

    exact_score_accuracy = np.mean(
        (rounded_a == y_true_a.astype(int)) &
        (rounded_b == y_true_b.astype(int))
    )

    outcome_from_score = np.where(
        rounded_a > rounded_b, "team_a_win",
        np.where(rounded_a < rounded_b, "team_b_win", "draw")
    )
    actual_outcome = test_data["result"].values
    rounded_score_outcome_accuracy = np.mean(outcome_from_score == actual_outcome)

    total_goals_mae = mean_absolute_error(
        y_true_a + y_true_b,
        pred_a + pred_b
    )

    mean_abs_goal_error_per_match = np.mean(
        np.abs(y_true_a - pred_a) + np.abs(y_true_b - pred_b)
    )

    return {
        "model": model_name,
        "team_a_mae": mae_a,
        "team_b_mae": mae_b,
        "mean_mae": (mae_a + mae_b) / 2,
        "team_a_rmse": rmse_a,
        "team_b_rmse": rmse_b,
        "mean_rmse": (rmse_a + rmse_b) / 2,
        "team_a_r2": r2_a,
        "team_b_r2": r2_b,
        "mean_r2": (r2_a + r2_b) / 2,
        "total_goals_mae": total_goals_mae,
        "mean_abs_goal_error_per_match": mean_abs_goal_error_per_match,
        "rounded_exact_score_accuracy": exact_score_accuracy,
        "rounded_score_outcome_accuracy": rounded_score_outcome_accuracy,
    }


results = []
predictions_by_model = {}
trained_models = {}

for model_name, estimator in regression_models.items():
    print("\nTraining:", model_name)

    model_a = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", clone(estimator)),
        ]
    )

    model_b = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", clone(estimator)),
        ]
    )

    model_a.fit(X_train, y_train_a)
    model_b.fit(X_train, y_train_b)

    pred_a = np.clip(model_a.predict(X_test), 0, None)
    pred_b = np.clip(model_b.predict(X_test), 0, None)

    metrics = evaluate_regression(y_test_a, pred_a, y_test_b, pred_b, model_name)

    results.append(metrics)
    predictions_by_model[model_name] = (pred_a, pred_b)
    trained_models[model_name] = (model_a, model_b)

comparison_df = pd.DataFrame(results).sort_values("mean_mae").reset_index(drop=True)

display(comparison_df)

comparison_df.to_csv(TABLES_DIR / "goal_regression_model_comparison.csv", index=False)
print("Saved:", TABLES_DIR / "goal_regression_model_comparison.csv")


## Presentation Plot 1 — MAE and RMSE Comparison

Lower MAE and RMSE indicate better goal prediction performance.

In [ ]:
plot_df = comparison_df.set_index("model")

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(plot_df.index))
width = 0.35

ax.bar(x - width/2, plot_df["mean_mae"], width, label="Mean MAE")
ax.bar(x + width/2, plot_df["mean_rmse"], width, label="Mean RMSE")

ax.set_xticks(x)
ax.set_xticklabels(plot_df.index, rotation=25, ha="right")
ax.set_ylabel("Error")
ax.set_title("Goal Regression Models — MAE and RMSE")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_mae_rmse_model_comparison.png", dpi=150)
plt.show()


## Presentation Plot 2 — Model Ranking by MAE

This chart ranks models from weakest to strongest according to mean MAE.

In [ ]:
rank_plot = comparison_df.sort_values("mean_mae", ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(rank_plot["model"], rank_plot["mean_mae"])
ax.set_title("Goal Regression Model Ranking by Mean MAE")
ax.set_xlabel("Mean MAE")
ax.set_ylabel("Model")

for i, value in enumerate(rank_plot["mean_mae"]):
    ax.text(value + 0.01, i, f"{value:.3f}", va="center")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "04_model_ranking_by_mean_mae.png", dpi=150)
plt.show()


## Presentation Plot 3 — R² Comparison

R² shows how much variance in goal counts is explained by the model.

In [ ]:
r2_plot = comparison_df.sort_values("mean_r2", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(r2_plot["model"], r2_plot["mean_r2"])
ax.set_title("Goal Regression Models — Mean R²")
ax.set_xlabel("Mean R²")
ax.set_ylabel("Model")

for i, value in enumerate(r2_plot["mean_r2"]):
    ax.text(value + 0.005, i, f"{value:.3f}", va="center")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "05_r2_model_comparison.png", dpi=150)
plt.show()


## Presentation Plot 4 — Exact Score Accuracy

Rounded predictions are used to estimate how often the exact final score is predicted.

In [ ]:
score_plot = comparison_df.sort_values("rounded_exact_score_accuracy", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(score_plot["model"], score_plot["rounded_exact_score_accuracy"])
ax.set_title("Rounded Exact Score Accuracy")
ax.set_xlabel("Exact score accuracy")
ax.set_ylabel("Model")

for i, value in enumerate(score_plot["rounded_exact_score_accuracy"]):
    ax.text(value + 0.002, i, f"{value:.3f}", va="center")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "06_exact_score_accuracy.png", dpi=150)
plt.show()


## Presentation Plot 5 — Outcome Accuracy from Rounded Scores

This plot checks whether rounded goal predictions produce the correct win/draw/loss outcome.

In [ ]:
outcome_plot = comparison_df.sort_values("rounded_score_outcome_accuracy", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(outcome_plot["model"], outcome_plot["rounded_score_outcome_accuracy"])
ax.set_title("Outcome Accuracy from Rounded Score Predictions")
ax.set_xlabel("Outcome accuracy")
ax.set_ylabel("Model")

for i, value in enumerate(outcome_plot["rounded_score_outcome_accuracy"]):
    ax.text(value + 0.005, i, f"{value:.3f}", va="center")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "07_outcome_accuracy_from_scores.png", dpi=150)
plt.show()


## Presentation Plot 6 — Total Goals Error

Total goals MAE measures how accurately each model predicts match scoring intensity.

In [ ]:
total_goal_plot = comparison_df.sort_values("total_goals_mae", ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(total_goal_plot["model"], total_goal_plot["total_goals_mae"])
ax.set_title("Total Goals MAE by Model")
ax.set_xlabel("Total goals MAE")
ax.set_ylabel("Model")

for i, value in enumerate(total_goal_plot["total_goals_mae"]):
    ax.text(value + 0.01, i, f"{value:.3f}", va="center")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "08_total_goals_mae_by_model.png", dpi=150)
plt.show()


## Best Model Selection

The best model is selected based on the lowest mean MAE.

In [ ]:
best_model_name = comparison_df.iloc[0]["model"]
best_pred_a, best_pred_b = predictions_by_model[best_model_name]
best_model_a, best_model_b = trained_models[best_model_name]

print("Best goal regression model:", best_model_name)

joblib.dump(best_model_a, MODELS_DIR / "best_comparison_team_a_goals_model.joblib")
joblib.dump(best_model_b, MODELS_DIR / "best_comparison_team_b_goals_model.joblib")

best_summary = comparison_df[comparison_df["model"] == best_model_name].copy()
display(best_summary)

best_summary.to_csv(TABLES_DIR / "best_goal_regression_model_summary.csv", index=False)

print("Saved:", MODELS_DIR / "best_comparison_team_a_goals_model.joblib")
print("Saved:", MODELS_DIR / "best_comparison_team_b_goals_model.joblib")
print("Saved:", TABLES_DIR / "best_goal_regression_model_summary.csv")


## Presentation Plot 7 — Actual vs Predicted Goals for Team A

This plot shows how close the best model is to real Team A goal counts.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test_a, best_pred_a, alpha=0.6)
min_val = min(y_test_a.min(), best_pred_a.min())
max_val = max(y_test_a.max(), best_pred_a.max())
ax.plot([min_val, max_val], [min_val, max_val])
ax.set_title(f"Actual vs Predicted Goals — Team A ({best_model_name})")
ax.set_xlabel("Actual goals")
ax.set_ylabel("Predicted goals")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "09_actual_vs_predicted_team_a.png", dpi=150)
plt.show()


## Presentation Plot 8 — Actual vs Predicted Goals for Team B

This plot shows how close the best model is to real Team B goal counts.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test_b, best_pred_b, alpha=0.6)
min_val = min(y_test_b.min(), best_pred_b.min())
max_val = max(y_test_b.max(), best_pred_b.max())
ax.plot([min_val, max_val], [min_val, max_val])
ax.set_title(f"Actual vs Predicted Goals — Team B ({best_model_name})")
ax.set_xlabel("Actual goals")
ax.set_ylabel("Predicted goals")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "10_actual_vs_predicted_team_b.png", dpi=150)
plt.show()


## Presentation Plot 9 — Actual vs Predicted Total Goals

Total goals are useful for explaining whether the model captures match intensity.

In [ ]:
actual_total_goals = y_test_a + y_test_b
predicted_total_goals = best_pred_a + best_pred_b

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(actual_total_goals, predicted_total_goals, alpha=0.6)
min_val = min(actual_total_goals.min(), predicted_total_goals.min())
max_val = max(actual_total_goals.max(), predicted_total_goals.max())
ax.plot([min_val, max_val], [min_val, max_val])
ax.set_title(f"Actual vs Predicted Total Goals ({best_model_name})")
ax.set_xlabel("Actual total goals")
ax.set_ylabel("Predicted total goals")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "11_actual_vs_predicted_total_goals.png", dpi=150)
plt.show()


## Presentation Plot 10 — Residual Distribution

Residuals show whether the selected model tends to overpredict or underpredict goals.

In [ ]:
residuals_a = y_test_a - best_pred_a
residuals_b = y_test_b - best_pred_b

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(residuals_a, bins=25, alpha=0.7, label="Team A residuals")
ax.hist(residuals_b, bins=25, alpha=0.7, label="Team B residuals")
ax.set_title(f"Goal Prediction Residuals — {best_model_name}")
ax.set_xlabel("Residual")
ax.set_ylabel("Frequency")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "12_goal_prediction_residuals.png", dpi=150)
plt.show()


## Presentation Plot 11 — Absolute Goal Error Distribution by Model

This boxplot compares the distribution of absolute goal errors across models.

In [ ]:
error_records = []

for model_name, (pred_a, pred_b) in predictions_by_model.items():
    abs_error_per_match = np.abs(y_test_a - pred_a) + np.abs(y_test_b - pred_b)

    for error in abs_error_per_match:
        error_records.append({
            "model": model_name,
            "absolute_goal_error": error,
        })

error_df = pd.DataFrame(error_records)

ordered_models = comparison_df["model"].tolist()
error_data = [
    error_df[error_df["model"] == model]["absolute_goal_error"].values
    for model in ordered_models
]

fig, ax = plt.subplots(figsize=(11, 6))
ax.boxplot(error_data, labels=ordered_models, vert=True)
ax.set_title("Absolute Goal Error Distribution by Model")
ax.set_ylabel("Absolute goal error per match")
ax.set_xlabel("Model")
plt.xticks(rotation=25, ha="right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "13_absolute_goal_error_boxplot.png", dpi=150)
plt.show()

error_df.to_csv(TABLES_DIR / "absolute_goal_error_by_model.csv", index=False)
print("Saved:", TABLES_DIR / "absolute_goal_error_by_model.csv")


## Presentation Plot 12 — Worst Prediction Errors

The largest prediction errors are reviewed to understand difficult matches.

In [ ]:
test_prediction_table = test_data[[
    "date", "team_a", "team_b", "tournament",
    "team_a_goals", "team_b_goals", "result"
]].copy()

test_prediction_table["best_model"] = best_model_name
test_prediction_table["pred_team_a_goals"] = best_pred_a
test_prediction_table["pred_team_b_goals"] = best_pred_b
test_prediction_table["pred_team_a_goals_rounded"] = np.round(best_pred_a).astype(int)
test_prediction_table["pred_team_b_goals_rounded"] = np.round(best_pred_b).astype(int)
test_prediction_table["predicted_score"] = (
    test_prediction_table["pred_team_a_goals_rounded"].astype(str)
    + " - "
    + test_prediction_table["pred_team_b_goals_rounded"].astype(str)
)
test_prediction_table["actual_score"] = (
    test_prediction_table["team_a_goals"].astype(int).astype(str)
    + " - "
    + test_prediction_table["team_b_goals"].astype(int).astype(str)
)
test_prediction_table["absolute_goal_error"] = (
    np.abs(test_prediction_table["team_a_goals"] - test_prediction_table["pred_team_a_goals"])
    + np.abs(test_prediction_table["team_b_goals"] - test_prediction_table["pred_team_b_goals"])
)

worst_predictions = test_prediction_table.sort_values(
    "absolute_goal_error",
    ascending=False
).head(15)

display(worst_predictions[[
    "date", "team_a", "team_b", "tournament",
    "actual_score", "predicted_score", "absolute_goal_error"
]])

worst_plot = worst_predictions.sort_values("absolute_goal_error", ascending=True).copy()
worst_plot["match"] = worst_plot["team_a"] + " vs " + worst_plot["team_b"]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(worst_plot["match"], worst_plot["absolute_goal_error"])
ax.set_title("Largest Goal Prediction Errors")
ax.set_xlabel("Absolute goal error")
ax.set_ylabel("Match")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "14_largest_goal_prediction_errors.png", dpi=150)
plt.show()

test_prediction_table.to_csv(TABLES_DIR / "best_model_test_predictions.csv", index=False)
worst_predictions.to_csv(TABLES_DIR / "worst_goal_prediction_errors.csv", index=False)
print("Saved:", TABLES_DIR / "best_model_test_predictions.csv")
print("Saved:", TABLES_DIR / "worst_goal_prediction_errors.csv")


## Presentation Plot 13 — Feature Importance for Best Tree Model

If the best model is tree-based, feature importance is exported and visualized.

In [ ]:
tree_model = best_model_a.named_steps["model"]
tree_preprocessor = best_model_a.named_steps["preprocessor"]

if hasattr(tree_model, "feature_importances_"):
    feature_names = tree_preprocessor.get_feature_names_out()
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": tree_model.feature_importances_,
    }).sort_values("importance", ascending=False)

    display(importance_df.head(20))
    importance_df.to_csv(TABLES_DIR / "best_tree_model_feature_importance.csv", index=False)

    top_importance = importance_df.head(20).sort_values("importance", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top_importance["feature"], top_importance["importance"])
    ax.set_title(f"Top 20 Feature Importances — {best_model_name}")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "15_best_model_feature_importance.png", dpi=150)
    plt.show()

    print("Saved:", TABLES_DIR / "best_tree_model_feature_importance.csv")
    print("Saved:", FIGURES_DIR / "15_best_model_feature_importance.png")
else:
    print(f"{best_model_name} does not provide feature_importances_.")
    print("Feature importance plot skipped for this selected model.")


## Final Output Overview

All presentation-ready regression comparison outputs are saved in one separate folder.

In [ ]:
print("Tables:")
print(" - output/regression_model_comparison_presentation/tables/goal_regression_model_comparison.csv")
print(" - output/regression_model_comparison_presentation/tables/best_goal_regression_model_summary.csv")
print(" - output/regression_model_comparison_presentation/tables/best_model_test_predictions.csv")
print(" - output/regression_model_comparison_presentation/tables/worst_goal_prediction_errors.csv")
print(" - output/regression_model_comparison_presentation/tables/absolute_goal_error_by_model.csv")
print(" - output/regression_model_comparison_presentation/tables/best_tree_model_feature_importance.csv, if available")

print("\nFigures:")
print(" - 01_total_goals_distribution.png")
print(" - 02_average_goals_per_year.png")
print(" - 03_mae_rmse_model_comparison.png")
print(" - 04_model_ranking_by_mean_mae.png")
print(" - 05_r2_model_comparison.png")
print(" - 06_exact_score_accuracy.png")
print(" - 07_outcome_accuracy_from_scores.png")
print(" - 08_total_goals_mae_by_model.png")
print(" - 09_actual_vs_predicted_team_a.png")
print(" - 10_actual_vs_predicted_team_b.png")
print(" - 11_actual_vs_predicted_total_goals.png")
print(" - 12_goal_prediction_residuals.png")
print(" - 13_absolute_goal_error_boxplot.png")
print(" - 14_largest_goal_prediction_errors.png")
print(" - 15_best_model_feature_importance.png, if available")

print("\nModels:")
print(" - output/regression_model_comparison_presentation/models/best_comparison_team_a_goals_model.joblib")
print(" - output/regression_model_comparison_presentation/models/best_comparison_team_b_goals_model.joblib")

print("\nPresentation-ready goal regression comparison completed.")
